# Serving-engine power trace: L0 + L1 + mixed batching

Runs entirely from GitHub. Nothing here needs a GPU.

**What this notebook adds** to the dynamic-shape simulator:

| layer | what it decides | ported from |
|---|---|---|
| **L0 arrivals** | when the next request shows up | Vidur (`static`, `poisson`, `gamma`, `trace`) |
| **L0 lengths** | how big it is | Vidur (`fixed`, `uniform`, `zipf`, `trace`) |
| **L1 scheduler** | who is in this batch | FSTS / vLLM V1 decode-first chunked prefill |
| **L1 KV** | who gets memory, and who loses it | Vidur block allocator, watermark, preemption |
| **L2 mixed** | what kernels that batch launches | new -- fuses the linear layers, keeps attention per request |

Multi-replica routing is deliberately out: one replica, as asked.

**The headline change.** Prefill and decode now happen in the *same* forward pass.
Every iteration under load carries a band of decode tokens with prefill filling
whatever token budget survives above it.

**Read the backend stamp on every figure.** With no LUT downloaded the predictor is
a roofline stand-in labelled `SYNTHETIC`. It gets trends right and absolute numbers
roughly. It is never a measurement.

## 1 - Install from GitHub

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'

if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=True)

if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'pytest'], check=True)

import dynshape
print('dynshape', dynshape.__version__, 'from', DIR)
print(subprocess.run(['git', '-C', DIR, 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())

## 2 - Run the test suite first

This is the gate. Everything below assumes it passes.

The tests worth knowing about, because they are the ones that could have caught a
silent error:

- `test_mixed.py::test_a_single_prefill_iteration_is_exactly_the_old_expansion` --
  with one request there is nothing to fuse and nothing to reorder, so the new
  mixed path must reproduce the old `expand()` **field for field**.
- `test_a_single_decode_iteration_is_exactly_the_old_expansion` -- the same identity
  from the other side, and a genuine cross-check: the fused half comes from the
  *prefill* template at one token, the attention half from the *decode* template at
  the true context. They have to agree.
- `test_the_two_classifiers_are_exact_complements` -- what fuses is decided
  arithmetically (does the field depend only on `B*S`?) and cross-checked
  structurally (does it read the key axis?). A disagreement would mean the fused
  expansion is silently wrong somewhere.

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -45

## 3 - Build the pieces

The shape rewriter and the predictor are unchanged from the previous stage. What is
new is the classification of the 242 template entries into *fusible* and
*per-request*.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dynshape import (ShapeRewriter, build_predictor, mixed_report,
                      ModelConfig, HardwareConfig, MemoryPlanner, BlockAllocator)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

rw   = ShapeRewriter.from_dir('templates/gpt2')
pred = build_predictor(force_analytic=True)

print('template   :', rw.n_kernels('prefill'), 'entries, decode shapes are', rw.decode_source)
print('classifier :', mixed_report(rw))

**What the classifier is saying.** 194 of the 242 entries depend only on the token
count `B*S`, so they can be emitted **once** at the summed token count of the whole
batch. That is exact, not an approximation -- a linear layer sees a bag of token
rows and does not care which request each row came from.

The remaining 48 are attention: 12 blocks x 4 kernels. Attention cares about nothing
*but* which request each row came from, so those are emitted per request.

48 is the same number of entries the decode `+1` correction touched, which is a
useful independent confirmation of both findings.

## 4 - L0, half one: when do requests arrive?

Identical requests per second. Identical total energy. Very different **peak**.
And peak is what sizes a breaker -- so the simplest component in the whole stack is
among the largest levers on the headline number.

In [ ]:
from dynshape import StaticInterval, PoissonInterval, GammaInterval

QPS, N = 10.0, 4000
gens = {
    'static (metronome)':   StaticInterval(qps=QPS),
    'poisson':              PoissonInterval(qps=QPS, seed=0),
    'gamma cv=0.5 (calm)':  GammaInterval(qps=QPS, cv=0.5, seed=0),
    'gamma cv=3.0 (bursty)': GammaInterval(qps=QPS, cv=3.0, seed=0),
}

fig, axes = plt.subplots(len(gens), 1, figsize=(13, 7), sharex=True)
rows = []
for ax, (name, g) in zip(axes, gens.items()):
    t = np.cumsum([g.next_interval() for _ in range(N)])
    counts, edges = np.histogram(t, bins=np.arange(0, t[-1], 1.0))
    ax.fill_between(edges[:-1], counts, step='post', alpha=0.8)
    ax.set_ylabel('req/s', fontsize=8)
    ax.set_title(f'{name}   mean {N/t[-1]:.1f} qps, busiest second {counts.max()}',
                 fontsize=9, loc='left')
    ax.set_xlim(0, 60)
    rows.append({'generator': name, 'mean qps': N/t[-1],
                 'busiest second': counts.max(),
                 'peak / mean': counts.max() / counts.mean()})
axes[-1].set_xlabel('time (s)')
fig.suptitle('Same rate, different clumping -- the arrival generator is the peak-power lever', y=1.0)
plt.tight_layout(); plt.show()

display(pd.DataFrame(rows).round(2))

Real arrivals clump more than Poisson does, because people react to the same events.
For synthetic traffic `gamma` with `cv > 1` is closer to reality; a replayed trace is
closer still, and is the only one of the four with a genuinely time-varying rate.

That last point matters more than it sounds. Constant-rate fluctuations are
independent across GPUs and cancel as sqrt(N), so a constant-lambda facility looks
*artificially smoother the bigger it gets*. A real lambda(t) moves every GPU together
and does not cancel.

In [ ]:
from dynshape import piecewise_poisson

# A synthetic lambda(t) by thinning -- ten lines, and Vidur has no generator for it.
day = piecewise_poisson([0.3, 4, 8, 12, 5, 0.5], segment_s=60.0, duration_s=360.0, seed=1)
counts, edges = np.histogram(day.times, bins=np.arange(0, 360, 10))
plt.figure(figsize=(12, 2.6))
plt.fill_between(edges[:-1], counts / 10.0, step='post', alpha=0.85, color='#c0392b')
plt.ylabel('lambda(t)  (req/s)'); plt.xlabel('time (s)')
plt.title('Synthetic time-varying rate -- the lunchtime hump, in 10 lines of thinning')
plt.grid(alpha=0.25); plt.show()

## 5 - L0, half two: how big are they?

Where the arrival generator sets batch size (the M dimension of every GEMM), the
length distribution decides **which phase dominates** -- and that is the difference
between compute-bound and memory-bound.

In [ ]:
from dynshape import UniformLength, ZipfLength, TrafficConfig, generate_traffic, traffic_summary

u = UniformLength(64, 4096, seed=0)
z = ZipfLength(64, 4096, theta=0.9, seed=0)
u_tot = np.array([sum(u.next_lengths()) for _ in range(4000)])
z_tot = np.array([sum(z.next_lengths()) for _ in range(4000)])

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].hist(u_tot, bins=60, alpha=0.85, color='#e67e22')
ax[0].set_title(f'uniform -- {100*(u_tot>2048).mean():.0f}% of requests exceed 2048 tokens')
ax[1].hist(z_tot, bins=60, alpha=0.85, color='#1f4e79')
ax[1].set_title(f'zipf theta=0.9 -- {100*(z_tot>2048).mean():.1f}% exceed 2048 tokens')
for a in ax:
    a.set_xlabel('total tokens'); a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print('Avoid uniform for power work: it invents far too many large requests, which')
print('inflates both KV pressure and the prefill share of the trace. The zipf tail is')
print('what actually fills the cache and triggers preemption.')

In [ ]:
rows = []
for name, ratio in [('write me a story', 0.5), ('balanced', 1.0),
                    ('ordinary chat', 4.0), ('summarise this doc', 19.0)]:
    s = traffic_summary(generate_traffic(TrafficConfig(
        num_requests=400, seed=0, prefill_to_decode_ratio=ratio)))
    rows.append({'workload': name, 'P:D': ratio,
                 'median P:D realised': round(s['pd_ratio_median'], 2),
                 'total prefill tokens': s['total_prefill_tokens'],
                 'total decode tokens': s['total_decode_tokens']})
display(pd.DataFrame(rows))
print('\nOne number decides whether the workload is compute-bound or memory-bound,')
print('and therefore roughly a factor of two in power. Same requests per second.')

### The seeding discipline

Vidur seeds once, globally, and four of its six random components then share that
stream. The run is reproducible -- that part is fine -- but it is not *attributable*.
Change the length generator and the arrival times move too, so an experiment meant to
compare two length distributions on identical traffic silently compares them on
different traffic.

Fixed here with `SeedSequence.spawn`: independent child streams, still fully
reproducible from one master seed.

In [ ]:
a = generate_traffic(TrafficConfig(num_requests=40, seed=9, length='zipf'))
b = generate_traffic(TrafficConfig(num_requests=40, seed=9, length='uniform'))
c = generate_traffic(TrafficConfig(num_requests=40, seed=9, interval='gamma', cv=2.5))

print('change the LENGTH generator  -> arrivals identical :',
      [r.arrived_at for r in a] == [r.arrived_at for r in b])
print('                             -> lengths differ     :',
      [r.num_prefill_tokens for r in a] != [r.num_prefill_tokens for r in b])
print('change the ARRIVAL generator -> lengths identical  :',
      [r.num_prefill_tokens for r in a] == [r.num_prefill_tokens for r in c])

## 6 - L1: sizing the KV pool

Vidur counts blocks; FSTS counts tokens. They are **algebraically identical** -- Vidur
just routes through a per-request worst case and multiplies it back out. The pool is
the same size in both; only the policy for spending it differs, and Vidur's policy is
the one that can preempt.

In [ ]:
model, hw = ModelConfig(), HardwareConfig()
planner = MemoryPlanner(model, hw, max_request_tokens=4096)
alloc   = BlockAllocator.from_memory(model, hw, 4096, block_size=16)

for k, v in planner.report().items():
    print(f'  {k:28s} {v}')
print()
print(f'  FSTS token capacity        {planner.kv_capacity_tokens():,.0f} tokens')
print(f'  Vidur block capacity       {alloc.capacity_tokens:,} tokens '
      f'({alloc.num_blocks:,} blocks x {alloc.block_size})')
print(f'  difference                 {planner.kv_capacity_tokens() - alloc.capacity_tokens:,.0f} '
      f'tokens, purely integer flooring')

**GPT-2 never runs out of KV on an A100.** Its cache costs 36 KiB per token, so the
pool holds about a million tokens -- the recompute spike simply cannot fire at this
model size. To *study* preemption we shrink the pool deliberately (section 11). That
is not a hack: it is how you reproduce, on a small model, the behaviour a 70B model
hits naturally.

## 7 - The run

### First: how much load does it take to make batching happen at all?

This decides everything below, and the intuitive answer is wrong by two orders of
magnitude.

GPT-2 is **fast**. A 280-token prefill is about 1 ms, a decode step about 0.8 ms, so a
typical request finishes in ~30 ms. Little's law says requests in flight = arrival rate
x time in system, so at 6 requests/second you get `6 x 0.03 = 0.18` concurrent
requests. The GPU runs **one request at a time and sits idle 78% of the run** -- there
is nothing to batch, and a trace taken there is a correct picture of an idle machine.

### And where is the ceiling?

Worth doing before picking a number, because the answer is not what a fused engine
would give. Under **eager** attention each request contributes its own 48 attention
kernels, so an iteration with N decoding requests launches `194 + 48N` kernels. Launch
overhead therefore grows **linearly with batch size**, which is exactly the cost that
batching is supposed to amortise away.

```
N =  1    242 kernels    ~0.5 ms of launch overhead    for  1 token
N = 10    674 kernels    ~1.3 ms                       for 10 tokens
N = 20  1,154 kernels    ~2.3 ms                       for 20 tokens
```

Per token that improves from 0.5 ms to 0.12 ms and then stops improving -- it is
asymptotically flat, not falling. A real varlen kernel would launch **one** attention
op for the whole batch and keep falling. This is the varlen gap from section 12,
showing up as a throughput ceiling rather than as an accuracy note.

So the demo runs at **100 qps**: about 10 requests in flight, comfortably below the
ceiling, and enough overlap for mixed batches to be the normal case rather than a
curiosity.

In [ ]:
from dynshape import EngineConfig, SchedulerConfig, run_engine, reset_ids
from dynshape.engine_plot import plot_engine_dashboard

reset_ids()
requests = generate_traffic(TrafficConfig(
    interval='gamma', qps=100.0, cv=2.0,    # bursty, and fast enough to overlap
    length='zipf', min_tokens=64, max_tokens=2048, theta=0.85,
    prefill_to_decode_ratio=4.0,            # ordinary chat
    num_requests=400, seed=0))

for k, v in traffic_summary(requests).items():
    print(f'  {k:24s} {v:,.2f}' if isinstance(v, float) else f'  {k:24s} {v:,}')

cfg = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                              block_size=16, max_tokens=4096),
    fuse_linear=True,
    record_kernels_until_ms=40.0)

trace = run_engine(requests, rw, pred, cfg, progress_every=500)

In [ ]:
s = trace.summary()
for k, v in s.items():
    print(f'  {k:28s} {v:,.4g}' if isinstance(v, float) else f'  {k:28s} {v}')

In [ ]:
fig = plot_engine_dashboard(trace, zoom_ms=40.0)
plt.show()

### How to read this

**Panel 1 (power).** Shaded stretches are genuine idle -- the engine has nothing to
run and is waiting for an arrival. The two horizontal lines are the two averages, and
the gap between them is the whole duty-cycle story: a report that quietly averages
only the busy segments overstates facility draw.

**Panel 2 (composition).** The blue band is decode tokens, one per running request,
taken off the top of the budget. Orange is prefill filling the remainder. Where both
appear, that is one forward pass doing both -- which is the thing this whole layer was
built to price.

Note the **shape** of the blue band. It is not the token budget that limits it; it is
how many requests happen to be mid-generation, which is set by the arrival rate and
nothing else. The orange spikes above it are single prompts arriving.

**Panel 3 (queue and KV).** On GPT-2 the KV line stays near zero -- 36 KiB per token
against a 40 GB card. That is correct, and it is the point of section 11.

**Panel 5 (energy).** Only the attention half is phase-attributable. A fused GEMM
genuinely belongs to prefill and decode at once -- that is *why* it was fused -- so it
gets its own bar rather than being split by an arbitrary rule.

**Panel 6 (kernel zoom).** The granularity the project exists for. The window opens at
the **first iteration**, not at t=0: under light load the engine can idle for hundreds
of milliseconds before the first request arrives, and a window anchored at zero would
close before any work happened and hand back an empty panel.

## 8 - Evidence that batches really are mixed

In [ ]:
idf, rdf, sdf = trace.to_dataframes()

mixed = idf[idf.is_mixed]
print(f'{len(mixed)} of {len(idf)} iterations ({100*len(mixed)/len(idf):.0f}%) '
      f'carried prefill and decode in the same forward pass\n')

cols = ['idx', 'decode_batch', 'prefill_chunks', 'prefill_tokens', 'decode_tokens',
        'total_tokens', 'context_mean', 'duration_ms', 'avg_power_w', 'n_kernels']
display(mixed[cols].head(12).round(2))

In [ ]:
def kind(row):
    if row.decode_batch > 0 and row.prefill_chunks == 0: return 'decode only'
    if row.decode_batch == 0 and row.prefill_chunks > 0: return 'prefill only'
    return 'mixed'

idf['kind'] = idf.apply(kind, axis=1)
summary = idf.groupby('kind').agg(
    iterations=('idx', 'size'),
    mean_power_W=('avg_power_w', 'mean'),
    mean_duration_ms=('duration_ms', 'mean'),
    mean_tokens=('total_tokens', 'mean'),
    energy_J=('energy_j', 'sum'))
summary['share of energy'] = summary.energy_J / summary.energy_J.sum()
display(summary.round(3))

print('Power tracks WHETHER prefill is present, not how much of the pass is prefill:')
print('a mixed iteration sits at the prefill-only power level, not between the two.')
print('Prefill is compute-bound and saturates the tensor cores as soon as it appears.')
print()
print('What chunking changes is DURATION and COUNT, not the height of the bar --')
print('mixed iterations are the longest of the three because they do both jobs, and')
print('there are far more of them, so the trace spends more time at a middling level')
print('instead of alternating between a spike and a floor.')

## 9 - Chunked vs unchunked

The clearest way to see what chunking does to a power trace. Same traffic, same GPU,
same total work -- one knob.

In [ ]:
from dynshape.engine_plot import plot_power_timeline, plot_batch_composition

def run_with(chunked, n=200, seed=3):
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='poisson', qps=100.0, length='zipf', min_tokens=128,
        max_tokens=2048, theta=0.8, num_requests=n, seed=seed))
    c = EngineConfig(scheduler=SchedulerConfig(
            chunk_size=512, max_num_seqs=256, max_tokens=4096,
            enable_chunked_prefill=chunked),
        record_kernels_until_ms=0.0)
    return run_engine(reqs, rw, pred, c)

on, off = run_with(True), run_with(False)

fig, axes = plt.subplots(2, 1, figsize=(13, 6.5))
plot_power_timeline(off, ax=axes[0], max_ms=off.total_time_ms)
axes[0].set_title('NO chunking -- a prefill owns its iteration: spike, quiet, spike, quiet')
plot_power_timeline(on, ax=axes[1], max_ms=on.total_time_ms)
axes[1].set_title('decode-first chunked prefill -- the same work, a much smoother ramp')
plt.tight_layout(); plt.show()

keys = ('iterations', 'mixed_fraction', 'wall_time_s', 'total_energy_j',
        'avg_power_w_busy', 'peak_iteration_power_w',
        'ttft_p50_s', 'ttft_p99_s', 'itl_p50_ms', 'itl_p99_ms')
cmp = pd.DataFrame({'no chunking': [off.summary()[k] for k in keys],
                    'chunked':     [on.summary()[k] for k in keys]},
                   index=['iterations', 'mixed fraction', 'wall time (s)', 'energy (J)',
                          'mean busy power (W)', 'peak iteration power (W)',
                          'TTFT p50 (s)', 'TTFT p99 (s)',
                          'ITL p50 (ms)', 'ITL p99 (ms)'])
display(cmp.round(3))

# The statistic chunking is actually for: how steady is the power draw?
for name, t in (('no chunking', off), ('chunked', on)):
    p = np.array([i.avg_power_w for i in t.iterations])
    print(f'{name:>12s}  iteration power: mean {p.mean():6.1f} W  '
          f'std {p.std():5.1f} W  p99 {np.percentile(p, 99):6.1f} W')

**Read both latency rows together, and in that order.**

TTFT gets *worse* with chunking and ITL gets *better*. That is not a defect, it is the
whole trade, and reporting only TTFT -- as the previous version of this notebook did --
guarantees chunking looks like a bad idea.

- **Existing users are protected.** Decodes come off the top of the budget every
  iteration, so nobody mid-sentence ever waits for a stranger's 2000-token prompt.
  That is the ITL row.
- **New users are sacrificed.** A newcomer's prompt is sliced into `chunk_size - (number
  of people decoding)` tokens per pass, so the busier the system the longer it takes to
  start. That is the TTFT row.

Losing your place mid-sentence feels worse than waiting slightly longer to start, which
is why production engines default to this.

For **power** the effect is the standard deviation, not the mean: same total work, same
energy, but the trace stops alternating between a compute-bound spike and a
memory-bound floor and settles into a band. Peak is what sizes a breaker, and the p99
iteration power is the number to watch.

## 10 - Burstiness against peak power

The lever, swept -- but **measured carefully**, because the obvious statistic does not
capture it.

`peak_iteration_power_w` is the power of the single hottest *iteration*, and that is
set by the largest single prefill chunk. Clumping does not make any one iteration
hotter; it makes **more of them happen at once**. So the metrics that move are peak
concurrency and peak power in a fixed time window.

Watch the busy-energy column. The naive expectation is that it stays flat -- it is the
same 300 requests with the same lengths every time, by construction, since the length
stream is seeded independently of the arrival stream. It does **not** stay flat, and
the reason is the interesting part.

In [ ]:
rows = []
for cv in [0.4, 0.7, 1.0, 1.5, 2.5, 4.0]:
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='gamma', qps=80.0, cv=cv, length='zipf', min_tokens=64,
        max_tokens=1024, theta=0.85, num_requests=300, seed=7))
    t = run_engine(reqs, rw, pred, EngineConfig(
        scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256, max_tokens=4096),
        record_kernels_until_ms=0.0))
    su = t.summary()
    # Peak power in a fixed 50 ms window -- what a meter with a real time
    # constant would see, and the thing clumping actually moves.
    W = 50.0
    bins = {}
    for i in t.iterations:
        b = int(i.t_start_ms // W)
        bins[b] = bins.get(b, 0.0) + i.energy_j
    peak_window_w = max(bins.values()) / (W / 1000.0) if bins else 0.0
    tokens = sum(r.num_generated_tokens for r in t.requests)
    rows.append({'cv': cv,
                 'busy energy (J)': t.busy_energy_j,
                 'mJ per token': 1000.0 * t.busy_energy_j / tokens,
                 'mean batch': np.mean([i.decode_batch for i in t.iterations]),
                 'peak concurrency': max(i.decode_batch for i in t.iterations),
                 'peak 50ms power (W)': peak_window_w,
                 'mean power wall (W)': su['avg_power_w_wallclock'],
                 'duty cycle': su['duty_cycle'],
                 'TTFT p99 (s)': su['ttft_p99_s']})
burst = pd.DataFrame(rows)
display(burst.round(3))

lo, hi = burst.iloc[0], burst.iloc[-1]
print(f"busy energy falls {100*(1 - hi['busy energy (J)']/lo['busy energy (J)']):.0f}% "
      f"from cv={lo.cv} to cv={hi.cv}, on identical requests.")
print()
print('Clumping forces LARGER BATCHES (mean batch '
      f"{lo['mean batch']:.1f} -> {hi['mean batch']:.1f}), and the 194 fused kernels --")
print('which re-read all 248 MB of weights every single pass -- are then amortised over')
print('more tokens. So bursty traffic is CHEAPER per token, and simultaneously worse for')
print('peak power and for tail latency. A real three-way trade, not a measurement error.')

fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
ax[0].plot(burst.cv, burst['peak concurrency'], 'o-', color='#1f4e79')
ax[0].set_ylabel('requests decoding at once'); ax[0].set_title('Clumping raises concurrency')
ax[1].plot(burst.cv, burst['mJ per token'], 'o-', color='#16a085')
ax[1].set_ylabel('mJ / output token'); ax[1].set_title('...which makes tokens cheaper')
ax[2].plot(burst.cv, burst['peak 50ms power (W)'], 'o-', color='#c0392b', label='peak 50 ms')
ax[2].plot(burst.cv, burst['mean power wall (W)'], 's--', color='#888', label='mean')
ax[2].set_ylabel('W'); ax[2].legend(fontsize=8); ax[2].set_title('...and peak diverge from mean')
ax[3].plot(burst.cv, burst['TTFT p99 (s)'], 'o-', color='#e67e22')
ax[3].set_ylabel('s'); ax[3].set_title('...and the tail pays for it')
for a in ax:
    a.set_xlabel('gamma cv (burstiness)'); a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 11 - Preemption: the recompute spike

Shrink the KV pool until it cannot hold every running request, and Vidur's allocator
does what a real engine does: pick a victim, throw away its cache, and re-prefill
everything it had generated.

Real GPU work, real watts, **no new output**, and nothing in the arrival pattern
predicts it. FSTS cannot produce this event at all -- it has no preemption -- which is
the main reason the block accounting here comes from Vidur.

In [ ]:
from dynshape import SimRequest
from dynshape.engine_plot import plot_queue_and_kv

reset_ids()
starved = [SimRequest(arrived_at=i * 0.02, num_prefill_tokens=400,
                      num_decode_tokens=400) for i in range(8)]
demand = sum(r.num_prefill_tokens + r.num_decode_tokens for r in starved)

cfg_small = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=512, max_num_seqs=32, block_size=16,
                              num_blocks=150, max_tokens=4096),   # 2400 tokens
    record_kernels_until_ms=0.0)
cfg_small.max_iterations = 200000

tp = run_engine(starved, rw, pred, cfg_small)
sp = tp.summary()
pool = cfg_small.scheduler.num_blocks * 16

print(f'pool            {pool:,} tokens')
print(f'demand          {demand:,} tokens  ({demand/pool:.1f}x oversubscribed)')
print(f'preemptions     {sp["preemptions"]}')
print(f'wasted work     {sp["restart_work_tokens"]:,} prompt tokens re-executed')
print(f'restarts        {sum(r.num_restarts for r in starved)} across {len(starved)} requests')
useful = sum(r.num_prefill_tokens for r in starved)
print(f'overhead        {100*sp["restart_work_tokens"]/useful:.0f}% extra prefill work,')
print(f'                producing no output at all')

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
plot_power_timeline(tp, ax=axes[0])
plot_queue_and_kv(tp, ax=axes[1])
plt.tight_layout(); plt.show()

Red vertical lines in the power panel mark preemptions. Notice where they sit:
against the KV line hitting its ceiling in the panel below, **not** against anything
in the arrival stream. That is the whole reason this event has to be simulated rather
than derived from the workload.

## 12 - Fusing vs concatenating

The v1 design proposed emitting one kernel list per request and concatenating them.
For the linear layers that is strictly worse than fusing, and the difference is not
subtle: each unfused GEMM re-reads the **same weights** from HBM.

Attention is the half that genuinely cannot be fused here. vLLM issues one ragged
`flash_attn_varlen_func` launch whose shape is a pair of offset arrays; EnergAIzer's
LUT is keyed on rectangles and has no varlen entry. Under eager attention there is
nothing to lose, which is what makes this the self-consistent choice for v1.

In [ ]:
def run_fuse(flag, seed=11):
    reset_ids()
    reqs = generate_traffic(TrafficConfig(
        interval='poisson', qps=80.0, length='zipf', min_tokens=128,
        max_tokens=1024, theta=0.8, num_requests=150, seed=seed))
    c = EngineConfig(scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                                               max_tokens=4096),
                     fuse_linear=flag, record_kernels_until_ms=0.0)
    return run_engine(reqs, rw, pred, c)

f, c = run_fuse(True), run_fuse(False)
cmp2 = pd.DataFrame({
    'fused': [sum(i.n_kernels for i in f.iterations), f.busy_time_ms,
              f.busy_energy_j, f.avg_busy_power_w],
    'concatenated': [sum(i.n_kernels for i in c.iterations), c.busy_time_ms,
                     c.busy_energy_j, c.avg_busy_power_w],
}, index=['kernels launched', 'busy time (ms)', 'busy energy (J)', 'mean busy power (W)'])
cmp2['ratio'] = cmp2.concatenated / cmp2.fused
display(cmp2.round(2))

mean_conc = np.mean([i.decode_batch + i.prefill_chunks for i in f.iterations])
print(f'mean batch size {mean_conc:.1f} requests')
print(f'predicted kernel ratio 242N / (194 + 48N) = '
      f'{242*mean_conc / (194 + 48*mean_conc):.2f}   observed {cmp2.loc["kernels launched", "ratio"]:.2f}')
print()
print('Same arithmetic, same FLOPs. Concatenating pays a launch, an HBM round trip and')
print('a low-occupancy tail per request instead of once per batch -- and note that time')
print('and energy scale with the KERNEL COUNT ratio, not with any change in the work.')

## 13 - Reporting power the way a meter would

Everything above reports power as **events**: one value per iteration, each a
different width. That is the natural output of a kernel simulator and it is useless
for three things you eventually want to do:

- **compare against a real capture.** NVML samples on a fixed grid. A variable-width
  staircase cannot be lined up against it without resampling first.
- **average across runs.** Two runs have different iteration boundaries, so there is
  no common x-axis to average over.
- **quote a peak.** A peak without an aperture is not a number -- see the sweep below.

FSTS reports a fixed-rate series instead, and that is the better choice. `resample()`
does the same here: a **box filter**, so the integral of the returned series equals
the trace's energy at any sample rate. The figure prints both so you can check it.

It also applies a one-pole **board response**. Assumption 5 of the design doc says a
real sensor never sees the square edges a kernel trace produces, because board
capacitance smooths microsecond transitions. That has been an assertion in a document
until now; the red line is it made visible.

In [ ]:
from dynshape.engine_plot import plot_resampled_power, plot_work_vector

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
plot_resampled_power(trace, dt_ms=1.0, smooth_tau_ms=5.0, ax=axes[0], max_ms=600)
axes[0].set_title(axes[0].get_title() + '   [first 600 ms]')
plot_resampled_power(trace, dt_ms=1.0, smooth_tau_ms=5.0, ax=axes[1], show_raw=False)
axes[1].set_title('The whole run at a 1 ms aperture')
plt.tight_layout(); plt.show()

### A peak is not a number until you say over what window

The same trace, resampled at different apertures. Nothing about the run changes -- only
how long the meter integrates for.

In [ ]:
rows = []
for dt in [0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]:
    t, p = trace.resample(dt_ms=dt)
    _, ps = trace.resample(dt_ms=dt, smooth_tau_ms=5.0)
    w = np.full(len(p), dt); w[-1] = trace.total_time_ms - (len(p)-1)*dt or dt
    rows.append({'aperture (ms)': dt, 'samples': len(p),
                 'peak (W)': p.max(), 'peak after board response (W)': ps.max(),
                 'mean (W)': float((p*w).sum()/w.sum()),
                 'integral (J)': float((p*w).sum()/1000.0)})
ap = pd.DataFrame(rows)
display(ap.round(3))

print(f"energy is conserved at every aperture: {ap['integral (J)'].min():.4f} to "
      f"{ap['integral (J)'].max():.4f} J against {trace.total_energy_j:.4f} J from the events.")
print(f"the mean is invariant too. The PEAK is not: {ap['peak (W)'].max():.0f} W at "
      f"{ap['aperture (ms)'].iloc[0]}ms down to {ap['peak (W)'].min():.0f} W at "
      f"{ap['aperture (ms)'].iloc[-1]}ms.")
print()
print('So a reported peak power needs its aperture attached, and the aperture should')
print('match whatever the number is for -- a breaker responds over milliseconds, a PDU')
print('reading averages over seconds, and they are not the same quantity.')

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.semilogx(ap['aperture (ms)'], ap['peak (W)'], 'o-', label='peak')
ax.semilogx(ap['aperture (ms)'], ap['peak after board response (W)'], 's--',
            label='peak after 5 ms board response')
ax.axhline(trace.avg_power_w, color='#888', ls=':', label='mean (invariant)')
ax.set_xlabel('meter aperture (ms)'); ax.set_ylabel('W')
ax.set_title('The same trace has different peaks at different apertures')
ax.legend(fontsize=8); ax.grid(alpha=0.25, which='both')
plt.tight_layout(); plt.show()

### The work vector: what is behind the watts

The other half of what FSTS reports. Per iteration, the FLOPs and bytes the shapes
imply -- **independent of any power model**, because they come from the geometry, not
from a prediction.

This is worth reporting beside the watts because it separates the two error sources.
If a predicted trace disagrees with a measured one, the work vector says whether the
simulator got the *work* wrong -- a scheduler or shape bug -- or the *conversion to
watts* wrong, which is a predictor bug. With only watts, those two are
indistinguishable, and with a SYNTHETIC predictor in the loop that distinction is the
whole game.

Note what is and is not split by phase. **FLOPs split exactly**: a fused GEMM's rows
each belong to one request and FLOPs are linear in the row count. **Bytes do not**:
the weight matrix is read once for the whole batch, which is the entire point of
fusing it. The code splits the first and refuses to invent a rule for the second.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.4))
plot_work_vector(trace, ax=ax, max_ms=600)
plt.tight_layout(); plt.show()

w = trace.work_totals()
tok = sum(r.num_generated_tokens for r in trace.requests)
print('work vector, summed over the run (no power model involved):')
print(f"  linear / MLP        {w['linear_flops']/1e12:9.2f} TFLOP   "
      f"{w['linear_bytes']/1e9:8.2f} GB")
print(f"  attention           {w['attn_flops']/1e12:9.2f} TFLOP   "
      f"{w['attn_bytes']/1e9:8.2f} GB")
print(f"  of which weights                      {w['weight_bytes']/1e9:8.2f} GB "
      f"({100*w['weight_bytes']/w['linear_bytes']:.0f}% of linear traffic)")
print()
print(f"  prefill             {w['prefill_flops']/1e12:9.2f} TFLOP")
print(f"  decode              {w['decode_flops']/1e12:9.2f} TFLOP")
print()
total_flops = w['linear_flops'] + w['attn_flops']
total_bytes = w['linear_bytes'] + w['attn_bytes']
print(f"  arithmetic intensity {total_flops/total_bytes:6.1f} FLOP/byte overall")
print(f"  {1000*trace.total_energy_j/tok:.2f} mJ per output token, "
      f"{total_flops/tok/1e9:.2f} GFLOP per output token")
print()
print('That weights line is the fusion argument in one number: most of the memory')
print('traffic in this workload is re-reading the model, not touching activations.')
print('Anything that makes a pass carry more tokens is amortising exactly that.')

## 14 - Export

Four tables: per iteration (now including the work vector), per request, per kernel
for the recorded window, and the fixed-rate power series.

In [ ]:
idf, rdf, sdf = trace.to_dataframes()
t_grid, p_grid = trace.resample(dt_ms=1.0)
_, p_smooth = trace.resample(dt_ms=1.0, smooth_tau_ms=5.0)
pdf = pd.DataFrame({'t_ms': t_grid, 'power_w': p_grid,
                    'power_w_board_5ms': p_smooth})

idf.to_csv('engine_iterations.csv', index=False)
rdf.to_csv('engine_requests.csv', index=False)
sdf.to_csv('engine_kernels.csv', index=False)
pdf.to_csv('engine_power_1ms.csv', index=False)
print('iterations', idf.shape, ' requests', rdf.shape,
      ' kernels', sdf.shape, ' power series', pdf.shape)
print('\nengine_power_1ms.csv is the one to compare against an NVML capture:')
print('uniform grid, energy-conserving, with and without a board response.')

try:
    from google.colab import files
    for f_ in ('engine_iterations.csv', 'engine_requests.csv',
               'engine_kernels.csv', 'engine_power_1ms.csv'):
        files.download(f_)
except Exception as e:
    print('(not on Colab, files left in', os.getcwd(), ')')

## What is still not modelled

Stated plainly, because these are the things that would change a number:

1. **Additivity.** Kernel costs are summed with no overlap, no memory-system
   contention, no cache carry-over. A mixed batch stresses this hardest, and it is
   physics rather than plumbing -- the only fix is measurement.
2. **HuggingFace kernels, vLLM schedule.** The scheduler assumes paged attention; the
   kernels come from an eager HF trace. The timeline and the kernels describe two
   different engines, so expect systematic over-prediction of decode time.
3. **No varlen attention.** Per-request rectangles instead of one ragged launch.
4. **Prefix caching is a static annotation.** A real engine discovers hits
   dynamically and evicts blocks under pressure; neither Vidur nor FSTS models that.
5. **One replica.** Routing is out of scope by request -- and it is the correlation
   knob that decides facility peak, so it matters later.
6. **The predictor is SYNTHETIC** until the EnergAIzer LUT is downloaded. Trends yes,
   absolute watts no.